# Exploring a YOLO-NAS embedding space in FiftyOne

YOLO-NAS spends nearly all of its compute building a visual representation, then throws it
away and keeps four numbers per object. This notebook keeps the representation instead.

We take the backbone features straight out of the model with `YoloNASEmbedder`, project
them to 2D with UMAP, and explore the result in the [FiftyOne](https://docs.voxel51.com/)
App. Lassoing a region of the plot selects those images in the sample grid, which is how
you actually find what a dataset contains: the near-duplicates, the mislabelled images,
the one class that turns out to be two, and the outliers worth deleting.

Nothing here is trained. These are the same COCO detection weights, read one stage earlier.

**What you'll do**

1. Load a dataset into FiftyOne
2. Embed every image with the YOLO-NAS backbone
3. Project to 2D and browse the embedding space
4. Find near-duplicates by cosine similarity
5. Embed individual *objects* (patches), not just whole images

## 1. Setup

FiftyOne and UMAP are optional extras — they are not needed to use `modern-yolonas`
itself, only for this notebook.

In [ ]:
# %pip install "modern-yolonas[fiftyone]" umap-learn

In [ ]:
import fiftyone as fo
import fiftyone.brain as fob
import fiftyone.zoo as foz
import numpy as np

from modern_yolonas import YoloNASEmbedder

print("fiftyone", fo.__version__)

## 2. A dataset

The FiftyOne Zoo's COCO-2017 validation split is a convenient stand-in and downloads in a
minute. Swap in your own data with `fo.Dataset.from_dir(...)` — everything below works on
any `fo.Dataset`, and the interesting version of this notebook is the one run on *your*
images, where you do not already know what is in them.

In [ ]:
dataset = foz.load_zoo_dataset(
    "coco-2017",
    split="validation",
    max_samples=500,      # raise this once you've seen it work end to end
    shuffle=True,
    seed=51,
    dataset_name="yolonas-embeddings",
    overwrite=True,
)
dataset.persistent = True
print(dataset)

## 3. Embed every image

`YoloNASEmbedder` returns L2-normalized vectors, so a dot product is already a cosine
similarity. The default `c5` layer is the deepest backbone map (post-SPP): the most
semantic representation in the network, 768-d on S, M and L alike.

The embedder pools only the non-padded region of the letterboxed canvas. That detail
matters here more than anywhere else — pooling the gray padding in would make this plot
cluster by aspect ratio instead of by content, and the result would look plausible while
telling you nothing.

In [ ]:
embedder = YoloNASEmbedder("yolo_nas_s", device="cuda")   # device="cpu" works, just slower
print(f"{embedder.embedding_dim}-d embeddings from layer {embedder.layers}")

In [ ]:
BATCH_SIZE = 16

paths = dataset.values("filepath")
embeddings = np.concatenate(
    [embedder.embed_batch(paths[i : i + BATCH_SIZE]) for i in range(0, len(paths), BATCH_SIZE)]
)

# Keep them on the samples so they survive the kernel and can be reused below.
dataset.set_values("yolonas_embedding", [e.tolist() for e in embeddings])
print(embeddings.shape)

### Try a wider descriptor

`c5` alone is a scene-level summary. Concatenating a mid-level map adds texture and part
structure, which separates visually similar scenes that differ in their details — at the
cost of a wider vector.

```python
embedder = YoloNASEmbedder("yolo_nas_s", layers=("c3", "c4", "c5"))   # 192 + 384 + 768 = 1344
```

Worth re-running the whole notebook with this once, to see how the map changes.

## 4. Project to 2D and explore

`compute_visualization` runs UMAP over the vectors we just computed. Launching the App
gives you the sample grid; open the **Embeddings** panel (the `+` next to the Samples tab)
and pick `yolonas_umap` to get the scatter plot.

Colour by `ground_truth.detections.label` and the structure becomes readable immediately:
if the backbone is doing its job, images sharing a dominant object land near each other
even though we never showed the projection a single label.

In [ ]:
results = fob.compute_visualization(
    dataset,
    embeddings=embeddings,
    method="umap",
    brain_key="yolonas_umap",
    seed=51,
)

In [ ]:
session = fo.launch_app(dataset)
# In the App: open the Embeddings panel, select `yolonas_umap`,
# colour by `ground_truth.detections.label`, then lasso a cluster.

**What to look for.** Lasso a tight knot of points and check the grid: a knot that is all
one scene type means the embedding is working. A knot that mixes two labels is worth
reading closely — it is usually either a genuine visual ambiguity or an annotation
mistake, and both are things you want to know about your data.

In [ ]:
# Whatever you lassoed in the App is available here as a view.
if session.selected:
    selected = dataset.select(session.selected)
    print(f"{len(selected)} samples selected")
    print(selected.count_values("ground_truth.detections.label"))

## 5. Near-duplicates

Because the vectors are normalized, the full cosine similarity matrix is one matrix
multiply. Pairs above ~0.98 are usually the same scene photographed twice, the same frame
sampled from a video, or a straight duplicate — all of which leak between train and
validation splits if you leave them in.

Tune the threshold on your own data rather than trusting this number: these features have
a high similarity floor (see the [embeddings guide](https://condadosai.github.io/modern-yolonas/guides/embeddings/)),
so the cut that means "duplicate" is dataset-specific.

In [ ]:
THRESHOLD = 0.98

similarity = embeddings @ embeddings.T
np.fill_diagonal(similarity, -1.0)          # never match a sample with itself

i, j = np.where(np.triu(similarity) > THRESHOLD)
pairs = sorted(zip(similarity[i, j], i, j), reverse=True)
print(f"{len(pairs)} pairs above {THRESHOLD}")

for score, a, b in pairs[:10]:
    print(f"  {score:.4f}  {paths[a].split('/')[-1]}  <->  {paths[b].split('/')[-1]}")

In [ ]:
# Inspect them in the App.
if pairs:
    duplicate_ids = {dataset.values("id")[k] for _, a, b in pairs for k in (a, b)}
    session.view = dataset.select(list(duplicate_ids))

## 6. Object embeddings, not image embeddings

Everything above describes whole images. For recognition and re-identification you want a
vector per *object*, and FiftyOne's patch views are built for exactly that.

`embed_boxes` takes boxes in the original image's pixel coordinates and crops them out of
the **feature maps** with `roi_align` — so every object in a frame costs one forward pass,
not one per crop.

Here the boxes come from the dataset's own annotations, so `embed_boxes` is the right
call. When the boxes have to be *detected* first, use the single-pass API in section 7
instead of running the model twice.

The result is an embedding space of things rather than scenes: cluster it and you get
"red buses", "people on bicycles", "close-up dog faces". This is the view to use when
building a re-identification gallery or auditing a single class.

In [ ]:
LABEL_FIELD = "ground_truth"

patch_embeddings = []
patch_ids = []

for sample in dataset.select_fields([LABEL_FIELD, "filepath", "metadata"]).iter_samples(progress=True):
    detections = sample[LABEL_FIELD].detections if sample[LABEL_FIELD] else []
    if not detections:
        continue

    height, width = sample.metadata.height, sample.metadata.width

    # FiftyOne stores boxes as relative [x, y, w, h]; embed_boxes wants absolute xyxy.
    xyxy = np.array(
        [
            [x * width, y * height, (x + w) * width, (y + h) * height]
            for x, y, w, h in (d.bounding_box for d in detections)
        ],
        dtype=np.float32,
    )

    patch_embeddings.append(embedder.embed_boxes(sample.filepath, xyxy))
    patch_ids.extend(d.id for d in detections)

patch_embeddings = np.concatenate(patch_embeddings)
print(patch_embeddings.shape, "object embeddings")

In [ ]:
patches = dataset.to_patches(LABEL_FIELD)

fob.compute_visualization(
    patches,
    embeddings=patch_embeddings,
    method="umap",
    brain_key="yolonas_patch_umap",
    seed=51,
)

session.view = patches
# Open the Embeddings panel again, select `yolonas_patch_umap`,
# and colour by `ground_truth.label`.

## 7. One pass for detections *and* embeddings

Everything so far embedded ground-truth boxes. On unlabelled data you need the model to
find the objects first — and there is no reason to run the backbone twice for that.

Detection and embedding share the whole network up to the head, so `predict` takes `Task`
flags and reads both off a single forward pass. This is the loop to use when indexing a
video or a large unlabelled set, where the second pass is the whole cost.


In [ ]:
from modern_yolonas import Task, YoloNASDetector

detector = YoloNASDetector("yolo_nas_s", device="cuda", conf_threshold=0.35)

result = detector.predict(
    dataset.first().filepath,
    Task.DETECT | Task.EMBED | Task.EMBED_OBJECTS,
)

print(f"{len(result.detections)} detections")
print("image embedding      ", result.embedding.shape)
print("per-object embeddings", result.detections.data["embedding"].shape)

# The vectors live in `data`, so supervision's slicing keeps the rows aligned.
people = result.detections[result.detections.class_id == 0]
print(f"{len(people)} people ->", people.data["embedding"].shape)


In [ ]:
# Index a whole unlabelled set: detections, object vectors and an image vector,
# one pass per image. Swap in your own file list.

records = []

for filepath in paths[:100]:
    result = detector.predict(filepath, Task.EMBED | Task.EMBED_OBJECTS)
    for box, class_id, confidence, vector in zip(
        result.detections.xyxy,
        result.detections.class_id,
        result.detections.confidence,
        result.detections.data.get("embedding", []),
    ):
        records.append(
            {
                "filepath": filepath,
                "label": detector.class_names[class_id],
                "confidence": float(confidence),
                "xyxy": box.tolist(),
                "embedding": vector,
            }
        )

print(f"{len(records)} object embeddings from {len(paths[:100])} images, one pass each")


## Where to go next

- **Index it.** These vectors drop straight into FAISS, Qdrant, LanceDB or
  `fob.compute_similarity` — the embedder's job ends at producing them.
- **Re-identification.** Pair `embed_boxes` with a supervision tracker and use the
  vectors as appearance features to re-link tracks across occlusions.
- **Curate a training set.** Sort by similarity to a known-hard example and label the
  neighbours first; they are the samples that will move your metric.
- **Know the limits.** These are detection features, not a metric-learning embedding —
  there is no contrastive objective behind them, so the ranking carries the signal and
  the absolute cosine value does not. The
  [embeddings guide](https://condadosai.github.io/modern-yolonas/guides/embeddings/)
  spells out what that means in practice.